In [11]:
print("Đang khai báo thư viện")

import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

print('Khai báo thư viện thành công')

Đang khai báo thư viện
Khai báo thư viện thành công


In [12]:
# Load test data
print('Khai báo dữ liệu')

test_path = os.path.join("Data", "test.csv")
df_test = pd.read_csv(test_path)
print(f"Test shape before processing: {df_test.shape}")

Khai báo dữ liệu
Test shape before processing: (3600, 17)


In [13]:
# Load params từ train
with open(os.path.join("Data", "Data-processed", "train_params.json")) as f:
    params = json.load(f)

print("\nLoaded train_params:")
print(json.dumps(params, indent=2))


Loaded train_params:
{
  "pop_median": 44.0,
  "key_mode": 7,
  "instr_median": 0.00399,
  "loudness_lo": -22.63567,
  "loudness_hi": -2.135,
  "duration_ms_lo": 112843.83251445,
  "duration_ms_hi": 526781.890014451,
  "tempo_lo": 70.97810499999999,
  "tempo_hi": 193.97530250000005
}


In [14]:
# ==================== 1. FIX DURATION MIXED UNITS ====================
mask_minutes = df_test['duration_in min/ms'] < 100
mask_ms = df_test['duration_in min/ms'] >= 100

print(f"\nValues in milliseconds : {mask_ms.sum():,} ({mask_ms.sum()/len(df_test)*100:.1f}%)")
print(f"Values in minutes      : {mask_minutes.sum():,} ({mask_minutes.sum()/len(df_test)*100:.1f}%)")

df_test['duration_in min/ms'] = df_test['duration_in min/ms'].apply(
    lambda x: x * 60 * 1000 if x < 100 else x
)
df_test.rename(columns={'duration_in min/ms': 'duration_ms'}, inplace=True)

print(f"duration_ms: min={df_test['duration_ms'].min()/60000:.2f} min, "
      f"max={df_test['duration_ms'].max()/60000:.2f} min")


Values in milliseconds : 3,095 (86.0%)
Values in minutes      : 505 (14.0%)
duration_ms: min=0.39 min, max=29.45 min


In [15]:
# ==================== 2. FIX LOUDNESS (giá trị dương) ====================
print("\nLoudness > 0 BEFORE fix:")
print(df_test[df_test['loudness'] > 0][['Id', 'loudness']])

df_test.loc[df_test['loudness'] > 0, 'loudness'] = -df_test.loc[df_test['loudness'] > 0, 'loudness']

print(f"Loudness range: [{df_test['loudness'].min():.3f}, {df_test['loudness'].max():.3f}]")
print(f"Values > 0 remaining: {(df_test['loudness'] > 0).sum()}")


Loudness > 0 BEFORE fix:
         Id  loudness
1092  15489     1.355
Loudness range: [-34.797, -1.101]
Values > 0 remaining: 0


In [16]:
# ==================== 3. IMPUTE MISSING VALUES (DÙNG PARAMS TỪ TRAIN) ====================
# Kiểm tra missing trước khi impute
missing_before = df_test.isnull().sum()
print(f"\nMissing before impute:\n{missing_before[missing_before > 0]}")

# Fill bằng giá trị từ train
df_test['Popularity'] = df_test['Popularity'].fillna(params['pop_median'])
df_test['key'] = df_test['key'].fillna(params['key_mode'])
df_test['instrumentalness'] = df_test['instrumentalness'].fillna(params['instr_median'])
df_test['key'] = df_test['key'].astype(int)

print(f"\nPopularity — filled with median from train = {params['pop_median']}")
print(f"key — filled with mode from train = {params['key_mode']}")
print(f"instrumentalness — filled with median from train = {params['instr_median']:.6f}")
print(f"\nMissing values remaining: {df_test.isnull().sum().sum()}")


Missing before impute:
Popularity           95
key                 405
instrumentalness    836
dtype: int64

Popularity — filled with median from train = 44.0
key — filled with mode from train = 7
instrumentalness — filled with median from train = 0.003990

Missing values remaining: 0


In [17]:
# ==================== 4. WINSORIZE (DÙNG BOUND TỪ TRAIN) ====================
def winsorize_test(df, col, lo, hi):
    before_min, before_max = df[col].min(), df[col].max()
    df[col] = df[col].clip(lo, hi)
    affected = ((df[col] == lo) | (df[col] == hi)).sum()
    print(f"{col:20s} before [{before_min:.3f}, {before_max:.3f}] "
          f"→ clipped to [{lo:.3f}, {hi:.3f}] | affected: {affected} rows")
    return df

print("\nWinsorizing (dùng bound từ train):")
print("-" * 75)
for col in ['loudness', 'duration_ms', 'tempo']:
    df_test = winsorize_test(df_test, col,
                             lo=params[f"{col}_lo"],
                             hi=params[f"{col}_hi"])


Winsorizing (dùng bound từ train):
---------------------------------------------------------------------------
loudness             before [-34.797, -1.101] → clipped to [-22.636, -2.135] | affected: 49 rows
duration_ms          before [23320.000, 1767000.000] → clipped to [112843.833, 526781.890] | affected: 69 rows
tempo                before [48.718, 214.396] → clipped to [70.978, 193.975] | affected: 75 rows


In [18]:
# ==================== 5. CYCLICAL ENCODING CHO KEY ====================
df_test['key_standard'] = df_test['key'] - 1
df_test['key_sin'] = np.sin(2 * np.pi * df_test['key_standard'] / 12)
df_test['key_cos'] = np.cos(2 * np.pi * df_test['key_standard'] / 12)
df_test.drop(columns=['key_standard'], inplace=True)

print("\nkey_sin, key_cos created")


key_sin, key_cos created


In [19]:
# ==================== 6. DROP UNNECESSARY COLUMNS ====================
drop_cols = ['Id', 'Artist Name', 'Track Name']
existing_drop = [col for col in drop_cols if col in df_test.columns]
df_test.drop(columns=existing_drop, inplace=True)

print(f"\nFinal shape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")


Final shape: (3600, 16)
Columns: ['Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature', 'key_sin', 'key_cos']


In [20]:

now = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"{now}_test_cleaned.csv"
filepath = os.path.join("Notebooks", "DataPreprocessing", filename)
os.makedirs(os.path.dirname(filepath), exist_ok=True)
df_test.to_csv(filepath, index=False)
print("\nSaved:", filepath)

fixed_filename = "test_cleaned.csv"
fixed_filepath = os.path.join("Data", "DataCleaned", fixed_filename)
df_test.to_csv(fixed_filepath, index=False)
print("Saved:", fixed_filepath)


Saved: Notebooks\DataPreprocessing\20260522_223243_test_cleaned.csv
Saved: Data\DataCleaned\test_cleaned.csv
